# Proof Of Concept Testing

## Test 1


Date : 14-Oct-2022

In [1]:
import pandas as pd
import os

In [2]:
data_file = os.path.join("..", "data", "nifty_data_2017_01_01_to_2022_09_19.csv")
print(data_file)

..\data\nifty_data_2017_01_01_to_2022_09_19.csv


In [3]:
df = pd.read_csv(data_file)
print(f"Data has {df.shape[0]} rows and {df.shape[1]} columns")
df.head()

Data has 96013 rows and 6 columns


,epoch datetime,open,high,low,close,volume
0,1500262800,9908.15,9908.15,9908.15,9908.15,0
1,1500263100,9908.15,9911.60,9899.50,9910.15,0
2,1500263400,9910.20,9910.20,9901.45,9907.65,0
3,1500263700,9907.45,9913.90,9906.05,9911.65,0
4,1500264000,9911.95,9915.15,9907.20,9909.00,0


In [4]:
df.dtypes

epoch datetime      int64
open              float64
high              float64
low               float64
close             float64
volume              int64
dtype: object

In [5]:
df["epoch datetime"] = pd.to_datetime(df["epoch datetime"], unit="s")

In [6]:
df = df.rename(columns = {
    "epoch datetime": "datetime",
    "open": "Open",
    "high": "High",
    "low": "Low",
    "close": "Close"
})

In [7]:
df.drop(["volume"], axis=1, inplace=True)

In [8]:
df = df.set_index("datetime")

In [9]:
df.head()

,Open,High,Low,Close
datetime,,,,
2017-07-17 03:40:00,9908.15,9908.15,9908.15,9908.15
2017-07-17 03:45:00,9908.15,9911.60,9899.50,9910.15
2017-07-17 03:50:00,9910.20,9910.20,9901.45,9907.65
2017-07-17 03:55:00,9907.45,9913.90,9906.05,9911.65
2017-07-17 04:00:00,9911.95,9915.15,9907.20,9909.00


## Test 2

In [10]:
from backtesting import Strategy
from backtesting.lib import crossover
from backtesting import Backtest

F:\AkashCode\Backtesting\.venv\lib\site-packages\backtesting\_plotting.py:50: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support (e.g. PyCharm, Spyder IDE). Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [31]:
import backtesting
backtesting.set_bokeh_output(notebook=False)

In [32]:
def SMA(values, n):
    """
    Compute SMA
    """
    return pd.Series(values).rolling(n).mean()

In [33]:
class SmaCross(Strategy):
    
    n1 = 10
    n2 = 20
    
    def init(self):
        # Precompute the two moving averages
        self.sma1 = self.I(SMA, self.data.Close, self.n1)
        self.sma2 = self.I(SMA, self.data.Close, self.n2)
    
    def next(self):
        # If sma1 crosses above sma2, close any existing
        # short trades, and buy the asset
        if crossover(self.sma1, self.sma2):
            self.position.close()
            self.buy()

        # Else, if sma1 crosses below sma2, close any existing
        # long trades, and sell the asset
        elif crossover(self.sma2, self.sma1):
            self.position.close()
            self.sell()

In [34]:
bt = Backtest(data=df, strategy=SmaCross, cash=100000, commission = 0.002)
stats = bt.run()

In [35]:
print(stats)

Start                     2017-07-17 03:40:00
End                       2022-09-19 09:55:00
Duration                   1890 days 06:15:00
Exposure Time [%]                   32.445606
Equity Final [$]                    9858.0509
Equity Peak [$]                      100000.0
Return [%]                         -90.141949
Buy & Hold Return [%]               77.952494
Return (Ann.) [%]                  -36.527018
Volatility (Ann.) [%]                6.885653
Sharpe Ratio                              0.0
Sortino Ratio                             0.0
Calmar Ratio                              0.0
Max. Drawdown [%]                  -90.143156
Avg. Drawdown [%]                  -90.143156
Max. Drawdown Duration     1890 days 03:55:00
Avg. Drawdown Duration     1890 days 03:55:00
# Trades                                 1673
Win Rate [%]                        18.230723
Best Trade [%]                      13.269955
Worst Trade [%]                     -8.106177
Avg. Trade [%]                    

In [38]:
bt.

AttributeError: 'Backtest' object has no attribute '_trades'